# Noun classification using skippy character n-grams

developed by Kow Kuroda (kow.kuroda@gmail.com)
created on 2024/12/13

Requirements
Python 3.11 or later is required for Neural Network classification to be successful. Note that Python 3.10 hangs at NN_model.fit (...).

Modifications
2025/02/15 implemented use of Morfessor segmentations as 1-grams

# Preparation

In [ ]:
## language selection

#target_langs_x         = [ 'French', 'OE', 'Gaelic', 'Italian', 'Spanish', 'Welsh' ]
## OE data is too small
target_langs          = [ 'Czech', 'French', 'German', 'Irish', 'Finnish' ]
target_lang           = target_langs[0]
print(f"target_lang: {target_lang}")

##
source_types          = [ 'spell', 'sound' ]
source_type           = source_types[0]
print(f"source_type: {source_type}")

In [ ]:
## settings
check             = False

## operational
max_doc_length    =  9 # longer doc takes longer time to process
min_doc_length    =  3
sample_size       = 2000
sample_n          = sample_size # alias
print(f"sample_n: {sample_n}")
reuse_sample      = False # retain the samples rather than renewing samples
balance_data      = False

In [ ]:
## doc settings
## orthography handling
uncapitalize          = True
print(f"uncapitalize: {uncapitalize}")

remove_diacritics     = True
print(f"remove_diacritics: {remove_diacritics}")

## boundary marking
mark_start            = False
mark_end              = False

if mark_end or mark_start:
    mark_boundaries   = True
else:
    mark_boundaries   = False
print(f"mark_boundaries: {mark_boundaries}")

## define hash as dependent dummy variable
if mark_boundaries:
    if mark_end and mark_start:
        hash_status  = "-hash-at-both"

    elif mark_end and not mark_start:
        hash_status  = "-hash-at-end"

    else:
        hash_status  = "-hashed-at-start"
else:
    hash_status      = "-no-hash"
print(f"hash_status: {hash_status}")

In [ ]:
## term settings

## use Morfessor segmentation or not
use_Morfessor        = True

## skippy{4,5}grams demand a longer time to complete
use_regex_to_split   = False
word_splitter        = r""

## n-grams
ngram_is_inclusive   = True
#inclusion_degree     = 1 # Valid values {1, 2, ... None}. None means truly recursive
print(f"ngram_is_inclusive: {ngram_is_inclusive}")

## term type: skippy-{4,5}-grams takes much time to generate
ngram_is_skippy      = True
n_for_ngram          = 2
if ngram_is_skippy:
    if n_for_ngram > 1:
        term_type = f"skippy{n_for_ngram}gram"
    else:
        term_type = f"{n_for_ngram}gram"
else:
    term_type = f"{n_for_ngram}gram"
print(f"term_type: {term_type}{hash_status}")

## skippy n-grams
gap_mark             = "…"
max_gap_ratio        = 0.100 # smaller values end earlier
max_gap_val          = round(max_doc_length * max_gap_ratio)
print(f"max_gap for skippy n-gram: {max_gap_val}")

In [ ]:
## DTM settings
import math
reduce_DTM            = True
min_term_freq         = round (math.sqrt(sample_n)/10) + 1
print(f"reduce_DTM: {reduce_DTM}")
print(f"min_term_freq: {min_term_freq}")

In [ ]:
## Other settings

## draw trees
draw_trees               = False

## use categorical encoding
use_numerical_encoding  = True
print(f"use_numerical_encoding: {use_numerical_encoding}")

## use supplement
use_supplement          = True
print(f"use_supplement: {use_supplement}")

## cross-validation
test_size_rate          = 0.1
print(f"test_size_rate [cross validation]: {test_size_rate}")

In [ ]:
## select data files
import glob
import pprint as pp

if   target_lang == 'Czech':
	data_dir = "Czech"
elif target_lang == 'German':
	data_dir = "German"
elif target_lang == 'Irish':
	data_dir = "Irish"
elif target_lang == 'Finnish':
	data_dir = "Finnish"
elif target_lang == 'French':
	data_dir = "French"
elif target_lang == 'OE':
	data_dir = "OE"
else:
	raise "Unknown data"
##
print(f"data_dir: {data_dir}")

##
data_files = glob.glob(f"data/inflected/{data_dir}/*")
data_files = sorted([ file for file in data_files if ".csv" in file ])
pp.pprint(data_files)

In [ ]:
## functions
def process_ske_lines (lines, form_dict_raw, form_counter, uncapitalize: bool = True, l_splitter = ",", f_splitter = ",", tag_cleaner = r'[*"]', word_reg = r"\w+", check: bool = False):
    "process Sketch Engine sample data to get token/pos pairs"
    import re
    counter = 0
    for i, line in enumerate (lines):
        if check:
            print(f"line {i:04d}")
        fields = line.split(l_splitter)
        if check:
            print(f"len(fields): {len(fields)}")
        if len (fields) < 6:
            continue
        ## main
        for field in fields:
            blocks = field.split (f_splitter)
            for block in blocks:
                if check:
                    print (f"block: {block}")
                try:
                    tokens = block.split ()
                    for token in tokens:
                        form, tag = token.split("/")
                        ## uncapitalize form
                        if uncapitalize:
                            form  = form.strip().lower()
                        else:
                            form  = form.strip()
                        ## process POS tag
                        tag       = re.sub (tag_cleaner, '', tag.strip())
                        if re.match (word_reg, form):
                            form_counter [form] += 1
                            ##
                            counter += 1
                            print (f"form {counter:04d} <{form}> with tag <{tag}> registered")
                            if tag not in form_dict_raw [form]: 
                                form_dict_raw [form].append (tag)
                except ValueError:
                    pass

In [ ]:
## process words in data
import io, re
import collections

form_counter  = collections.defaultdict(int)
form_dict_raw = collections.defaultdict(list)
check = True
for file in data_files:
    print(f"opening {file}")
    with io.open(file, encoding = 'utf-8_sig') as word:
        lines = word.readlines()
        process_ske_lines (lines, form_dict_raw = form_dict_raw, form_counter = form_counter)
##
form_dict_raw

In [ ]:
## check frequency
freq_forms = list(map(lambda x: f"{x[-1]} {x[0]}", sorted (form_counter.items(), key = lambda x: x[-1], reverse = True)))
freq_forms

# Set up data 1: Encoding grammatical attributes

In [ ]:
target_lang

In [ ]:
## regularize Irish dict
if target_lang == 'Irish':
    D = {}
    for k, vs in form_dict_raw.items():
        V = []
        for v in vs:
            W = []
            for x in v.split("|"):
                W.append(x)
        V.extend(W)
        D[k] = V
    ##
    if check:
        for k, v in D.items():
            print(f"{k} : {v}")
    ##
    form_dict_raw = D
##
form_dict_raw

In [ ]:
## filter out offensive words
remove_pat = r'[.-]'
form_dict_raw = { k : v for k, v in form_dict_raw.items() if not re.match(remove_pat, k) }
form_dict_raw

In [ ]:
## define form_dict: N.B. that different languages use different POS system
from utils import simplify
def parse_pos (tag: str, lang: str, i: int, check: bool = False):
    """analyze POS tags"""
    import re
    ## Czech
    if lang in [ 'Czech' ]:
        L = []; y = []
        for i, x in enumerate ([ x for x in re.split(r"", tag) if len(x) > 0 ]):
            if i % 2 == 1:
                y.append(x)
                L.append ("".join(y))
                y = []
            else:
                y.append(x)
    ## Irish, French
    elif lang in [ 'Irish', 'French' ]:
        L = [ x for x in re.split(r"", tag) if len(x) > 0 ]
            
    ## German
    elif lang in [ 'German' ]:
        L = [ x for x in tag.split(".") if len(x) > 0 ]
    
    ## Other
    else:
        L = [ x for x in tag.split(",") if len(x) > 0 ]
    ##
    if check:
        print(f"L: {L}")
    return simplify (L, nested = False)

In [ ]:
## segement POS tags and define form_dict
import collections
form_dict = collections.defaultdict(list)
check = False
for form, tags in form_dict_raw.items():
    A = []
    for i, tag in enumerate(tags):
        if check:
            print(f"tag: {tag}")
        a = parse_pos (tag, target_lang, i)
        if a not in A:
            A.append (a)
    #
    form_dict[form] = simplify (A, nested = False)
##
form_dict

In [ ]:
## POS mapping
## Czech
Czech_pos_renamer = { 'k1' : "Noun", 'k2' : 'Adj', 'k3' : 'Pron', 'k4' : 'Number', 'k5' : 'Verb',  'k6' : 'Adv', 'k7' : 'Prep', 'k8' : 'Conj', 'k9' : 'Part', 'k0' : 'Inter', 'kA' : 'Abbrev', 'kI' : 'Punct',
					 'gF' : 'Fem', 'gM': 'Masc0', 'gI': 'Masc1', 'gN': 'Neut',
					 'nS' : 'Sg', 'nP' : 'Pl',
					 'c1': "Nom", 'c2': 'Gen', 'c3': 'Dat', 'c4': 'Acc', 'c5': 'Voc', 'c6': 'Loc', 'c7' : 'Instr' }
##
def rename_Czech_pos (x: str):
	try:
		return Czech_pos_renamer [x]
	except KeyError:
		return x

## German
German_pos_renamer = { 'N': "Noun", 'PRO': 'Pron', 'V': 'Verb', 'ADJA': 'Adj', 'R': 'Adv', 'CONJ': 'Conj' }
##
def rename_German_pos (x: str):
	try:
		return German_pos_renamer [x]
	except KeyError:
		return x


## Irish
Irish_pos_renamer = { 'N': "Noun", 'V': 'Verb', 'A': 'Adj', 'R': 'Adv', 'P': 'Pron', 'f': 'Fem', 'm': "Masc", 's': 'Sg', 'p': 'Pl', 'v': 'Voc', 'g': "Gen", 'd': 'Dat', 'c': 'Nom', '-': '-', 'e': 'Emp' }
##
def rename_Irish_pos (x: str):
	try:
		return Irish_pos_renamer [x]
	except KeyError:
		return x

##
def pos_mapper (pos_map: dict, x: str):
	try:
		return pos_map [x]
	except KeyError:
		return x

## French
def French_pos_analyzer (L: list, null: str = 'x'):
	T = []
	sig = L[0]
	if sig == 'N':
		for i, x in enumerate (L):
			if   i == 0:
				T.append ('Noun')
			## process substantive encoding
			elif i == 1:
				if   x == 'C':
					T.append ('common')
				elif x == 'P':
					T.append ('proper')
				else:
					pass
			elif i == 2:
				if   x == 'F':
					T.append ('Fem')
				elif x == 'M':
					T.append ('Masc')
				elif x == 'C':
					T.append ('Comm')
				elif x == 'N':
					T.append ('Neu')
				else:
					pass
			elif i == 3:
				if   x == 'S':
					T.append ('Sg')
				elif x == 'P':
					T.append ('Pl')
				elif x == 'N':
					T.append ('Inv')
				else:
					pass
			elif i == 4:
				#if   x == 'S':
				#	T.append ('Person')
				#elif x == 'G':
				#	T.append ('Location')
				#elif x == 'O':
				#	T.append ('Organization')
				#else:
				#	pass
				pass
			elif i == 5:
				pass
			elif i == 6:
				if   x == 'A':
					T.append ('Aug')
				elif x == 'D':
					T.append ('Dim')
				else:
					pass
			else:
				pass

	elif sig == 'A':
		for i, x in enumerate (L):
			if i == 0:
				T.append ('Adj')
			else:
				pass
	elif sig == 'Adp':
		for i, x in enumerate (L):
			if i == 0:
				T.append ('Adj')
			else:
				pass
	elif sig == 'C':
		for i, x in enumerate (L):
			if i == 0:
				T.append ('Conj')
			else:
				pass
	elif sig == 'D':
		for i, x in enumerate (L):
			if i == 0:
				T.append ('Det')
			else:
				pass
	elif sig == 'I':
		for i, x in enumerate (L):
			if i == 0:
				T.append ('Int')
			else:
				pass
	elif sig == 'P':
		for i, x in enumerate (L):
			if i == 0:
				T.append ('Prep')
			else:
				pass
	elif sig == 'S':
		for i, x in enumerate (L):
			if i == 0:
				T.append ('Adp')
			else:
				pass
	elif sig == 'R':
		for i, x in enumerate (L):
			if i == 0:
				T.append ('Adv')
			else:
				pass
	elif sig == 'V':
		for i, x in enumerate (L):
			if i == 0:
				T.append ('Verb')
			else:
				pass
	elif sig == 'Z':
		for i, x in enumerate (L):
			if i == 0:
				T.append ('Num')
			else:
				pass
	else:
		pass

	##
	return T
			

In [ ]:
## convert POS tag
check = True
if target_lang in [ 'Czech', 'Irish', 'French' ]:
	form_dict_new = {}
	for word, tags in form_dict.items():
		if check:
			print(f"word: {word}; tags: {tags}")
		T = []
		if target_lang in ['French']:
			for tag in tags:
				if check:
					print(f"tag: {tag}")
				X = French_pos_analyzer (tag)
				if check:
					print (f"X: {X}")
				##
				T.append (X)
		else:
			for tag in tags:
				if check:
					print(f"tag: {tag}")
				X = []
				for seg in tag:
					if target_lang == 'Czech':
						#pos_new = rename_Czech_pos (seg)
						pos_new = pos_mapper (Czech_pos_renamer, seg)
						
					elif target_lang == 'German':
						#pos_new = rename_German_pos (seg)
						pos_new = pos_mapper (German_pos_renamer, seg)
					
					elif target_lang == 'Irish':
						#pos_new = rename_Irish_pos (seg)
						pos_new = pos_mapper (Irish_pos_renamer, seg)
					#
					X.append (pos_new)
				## Czech
				if target_lang == 'Czech':
					X = [ t for t in X if t in Czech_pos_renamer.values() ]
				## German
				elif target_lang == 'German':
					X = [ t for t in X if t in German_pos_renamer.values() ]
				## Irish
				elif target_lang == 'Irish':
					X = [ t for t in X if t in Irish_pos_renamer.values() ]
				if check:
					print (f"tag*: {X}")
				##
				T.append(X)
		##
		form_dict_new[word] = T
	##
	form_dict = form_dict_new
##
form_dict

In [ ]:
## define inflect_dict
inflect_dict = {}
if target_lang in [ 'German' ]:
    pos_list = [ 'N', 'A', 'V', 'Pro' ]
elif target_lang in [ 'Irish' ]:
    pos_list = [ 'N', 'A', 'V', 'Pro' ]
    pos_list_x = pos_list + [ 'R', 'D', 'Q', 'S', 'T', 'C', 'M' ]
else:
    pos_list = [ 'N', 'A', 'V', 'Pro' ]
print(f"pos_list: {pos_list}")
##
for pos in pos_list:
    print(f"processing: {pos}")
    X_form_dict = {}
    pos_pat = f'r"{pos}.*"'
    #X_form_dict = { k: v for k, v in form_dict.items() if isinstance(x, list) and re.match(eval(pos_pat), v[0][0]) }
    for k, v in form_dict.items():
        try:
            m = re.match(eval(pos_pat), v[0][0])
            X_form_dict[k] = v
        except IndexError:
            pass   
    inflect_dict[pos] = X_form_dict

## pos-wise dicts
N_inflect_dict_all   = inflect_dict['N']
A_inflect_dict_all   = inflect_dict['A']
V_inflect_dict_all   = inflect_dict['V']
Pro_inflect_dict_all = inflect_dict['Pro']
#X_inflect_dict_all   = inflect_dict['X'] 

## check result
import random
random.sample(list(N_inflect_dict_all.items()), 20)

In [ ]:
## create N_attribute_dict
import collections
N_inflect_dict   = collections.defaultdict(list)
N_attribute_dict = collections.defaultdict(int)
check = True
for k, vx in N_inflect_dict_all.items():
    for vs in vx:
        if check:
            print(f"vs: {vs}")
        ## filtering out irrelevant cases: non nouns and proper nouns
        if target_lang in ['German']:
            if vs[0] != 'N' or vs[1] != 'Reg':
                continue
            else:
                print(f"processing: {vs}")
                for v in vs:
                    N_attribute_dict[v] += 1
                N_inflect_dict[k] = vs
        elif target_lang in ['French']:
            if vs[0] != 'Noun' or vs[1] != 'common':
                continue
            else:
                print(f"processing: {vs}")
                for v in vs:
                    N_attribute_dict[v] += 1
                N_inflect_dict[k] = vs
        else:
            try:
                if vs[0] != 'Noun':
                    continue
                else:
                    print(f"processing: {vs}")
                    for v in vs:
                        N_attribute_dict[v] += 1
                    N_inflect_dict[k] = vs
            except IndexError:
                pass
##
N_attribute_dict

In [ ]:
## check result
import random
random.sample(list(N_inflect_dict.items()), 20)

# Attribute selection

In [ ]:
## get all attributes
N_attributes_all = list(N_attribute_dict.keys())
#N_attributes_all = [ x for x in N_attributes_all if x != "Noun" ]
N_attributes_all

In [ ]:
## select effective attributes
if target_lang in [ 'Czech' ]:
    gender_index    = [4,6,9,1]
    plurality_index = [2,10]
    case_index      = [5,8,7,12,3,11,13] # Nominative, Accusative, Dative, Genitiv

elif target_lang in [ 'French' ]:
    gender_index    = [4,2,6]
    plurality_index = [7,5,3]
    case_index      = []
    
elif target_lang in [ 'German' ]:
    gender_index    = [8,4,10]
    plurality_index = [3,9]
    case_index      = [5,2,7,6] # Nominative, Accusative, Dative, Genitiv

elif target_lang in [ 'Irish' ]:
    gender_index    = [7,4]
    plurality_index = [1,5]
    case_index      = [3,6,9,8] # Nom=Acc, Genitive, Dative, Vocative

#
effective_index = gender_index + plurality_index + case_index
if check:
    print(f"effective_index: {effective_index}")
##
N_attributes = [ N_attributes_all[i] for i in effective_index ]
N_attributes

In [ ]:
## functions
def encode_attributes (D: list, A: list, check: bool = False) -> dict:
    import collections
    M = collections.defaultdict(bool)
    for a in A:
        if a in D:
            M[a] = 1
        else:
            M[a] = 0
    ##
    return M

In [ ]:
## create encoded_N_inflect_dict
encoded_N_inflect_dict = collections.defaultdict(list)
if target_lang in ['German']:
    min_size = 5
else:
    min_size = 8
#
for k, vx in N_inflect_dict.items():
    print(f"processing: {k}; {vx}")
    ## fail-safe operation
    if len(vx) > min_size:
        for vs in vx:
            encoding = encode_attributes (vs, N_attributes)
            print(f"encoding: {encoding}")
            if len([ v for v in encoding.values() if v == True ]) > 0:
                encoded_N_inflect_dict[k].append(encoding)
            else:
                print(f"{k} failed encoding: {encoding}")
    ## normal operation
    else:
        encoding = encode_attributes (vx, N_attributes)
        print(f"encoding: {encoding}")
        if len([ v for v in encoding.values() if v == True ]) > 0:
            encoded_N_inflect_dict[k].append(encoding)
        else:
            print(f"{k} failed encoding: {encoding}")


In [ ]:
## check result
import random
random.sample(list(encoded_N_inflect_dict.items()), 10)

In [ ]:
## create encoded_N_inflect_df
import pandas as pd
full_df = pd.DataFrame()
#
if target_lang in ['German']:
    min_size = 5
else:
    min_size = 8
#
for k, vx in encoded_N_inflect_dict.items():
    if len(vs) > min_size:
        for vs in vx:
            dfx = pd.DataFrame(data = vs)
            full_df = pd.concat([full_df, dfx], ignore_index = True) # Crucially, ignore_index
    else:
        dfx = pd.DataFrame(data = vx)
        full_df = pd.concat([full_df, dfx], ignore_index = True)
##
#full_df.reset_index(drop = True)

## Czech case merger
merge_cases = True
if target_lang in ['Czech'] and merge_cases:
    full_df.insert(loc = 3, column = 'Masc', value = full_df['Masc0'] + full_df['Masc1'])

## add form column
full_df['form'] = encoded_N_inflect_dict.keys()

## check
full_df

# Filtering and sampling data

In [ ]:
## remove too long and too short words
import unicodedata
full_df['size'] = full_df['form'].apply(lambda x:
                                        len(unicodedata.normalize('NFC', x)))
full_df = full_df[ full_df['size'] <= max_doc_length ]
full_df = full_df[ full_df['size'] >= min_doc_length ]

In [ ]:
## set df from full_df
df = full_df.sample(sample_n)
df

# Build terms

In [ ]:
## uncapitalize
if uncapitalize:
	df.loc[:,'form'] = df['form'].apply(lambda x: str(x).lower())

In [ ]:
## add boundary symbols
mark_end         = False
mark_start       = False
##
if mark_end or mark_start:
	mark_boundaries  = True
else:
	mark_boundaries  = False
###
if mark_boundaries:
    if mark_end and mark_start:
        hash_status  = "-hash-at-both"

    elif mark_end and not mark_start:
        hash_status  = "-hash-at-end"

    else:
        hash_status  = "-hashed-at-start"
else:
    hash_status      = "-no-hash"
print(f"hash_status: {hash_status}")
##
if mark_boundaries:
	## avoid re-adding hashes
	hashed_test = df['form'].apply(lambda x: str(x)[0] == '#' and str(x)[-1] == "#")
	print(f"hashed_test: {all (hashed_test == True)}")
	if any (hashed_test == True):
		df.loc[:,'form'] = df['form'].apply(lambda x: x.strip('#'))
		if mark_end and mark_start:
			df.loc[:,'form'] = df['form'].apply(lambda x: f"#{str(x)}#")
		elif mark_end:
			df.loc[:,'form'] = df['form'].apply(lambda x: f"{str(x)}#")
		elif mark_start:
			df.loc[:,'form'] = df['form'].apply(lambda x: f"#{str(x)}")
		else:
			df.loc[:,'form'] = df['form'].apply(lambda x: x.strip('#'))
	else:
		if mark_end and mark_start:
			df.loc[:,'form'] = df['form'].apply(lambda x: f"#{x}#")
		elif mark_end and not mark_start:
			df.loc[:,'form'] = df['form'].apply(lambda x: f"{x}#")
		elif not mark_end and mark_start:
			df.loc[:,'form'] = df['form'].apply(lambda x: f"#{x}")
		else:
			pass
else:
	df.loc[:,'form'] = df['form'].apply(lambda x: x.strip('#'))

## check
df['form']

In [ ]:
## generate 1-gram
def re_split(splitter, x):
    import re
    return [ y for y in re.split(splitter, x) if len(y) > 0 ]

if use_Morfessor:
    import morfessor as morf
    mio = morf.MorfessorIO()
    morph_size = 2.1
    model_name = f"model-{target_lang}-l{morph_size}.bin"
    print(f"model_name: {model_name}")
    model_file = f"Morfessor/models/{model_name}"
    print(f"model_file: {model_file}")
    model = mio.read_binary_model_file(model_file)
    df['1gram'] = df['form'].apply(lambda x: model.segment(x))
else:
    conservative = True
    if use_regex_to_split:
        df.loc[:,'1gram'] = df['form'].apply(lambda x: re_split(word_splitter, x))
    else:
        import unicodedata
        if conservative:
            df['1gram'] = [ [ y for y in unicodedata.normalize('NFC', x) if len(y) > 0 ] for x in df['form'] ]
        else:
            ## The code above turned out to be ineffective since it separates diacritics
            df.loc[:,'1gram'] = [ [ *unicodedata.normalize('NFC', x) ] for x in df['form'] ]
#
df['1gram']

In [ ]:
## generic function for n-gram generation
import numpy as np
def add_ngram_to_df (dfx, n_for_ngram: int, prefix: str = "", skippy: bool = False, skippy_means_extended: bool = False, seg_joint: str = "", missing_mark: str = gap_mark, max_distance = None, inclusive: bool = ngram_is_inclusive, check: bool = False):
    """
    generic function for adding n-gram column to df with a specified n for ngram
    """
    ## set source_var
    source_var = f"{prefix}1gram"
    print(f"===================")
    print(f"source_var: {source_var}")
    unigrams = list(dfx[source_var]) # Crucially
    
    ## set target_var
    if skippy:
        if n_for_ngram > 1:
            target_var = f"{prefix}skippy{n_for_ngram}gram"
        else:
            target_var = f"{prefix}{n_for_ngram}gram"
    else:
        target_var = f"{prefix}{n_for_ngram}gram"
    print(f"target_var: {target_var}")
    
    ## (skippy) n-gram の生成
    import gen_ngrams
    if skippy:
        if skippy_means_extended:
            ngrams_inner = [ gen_ngrams.gen_extended_skippy_ngrams(x, n = n_for_ngram, sep = seg_joint, missing_mark = gap_mark, max_distance = max_distance, check = False) for x in unigrams ]
        else:
            ngrams_inner = [ gen_ngrams.gen_skippy_ngrams(x, n = n_for_ngram, sep = seg_joint, missing_mark = gap_mark, max_distance = max_distance, check = False) for x in unigrams ]
    else:
        ngrams_inner = [ gen_ngrams.gen_ngrams(x, n = n_for_ngram, sep = seg_joint, check = False) for x in unigrams ]
    if check:
        print(f"ngrams: {ngrams_inner}")
    
    ## 包括的 ngramの生成
    if inclusive:
        if skippy and n_for_ngram > 2:
            supplement_var = f"{prefix}skippy{n_for_ngram - 1}gram"
        else:
            supplement_var = f"{prefix}{n_for_ngram - 1}gram"
        print(f"supplement_var: {supplement_var}")
        ##
        for i, g in enumerate(ngrams_inner):
            supplement = [ x for x in list(dfx[supplement_var])[i] if len(x) < n_for_ngram and x not in g ]
            if check:
                print(f"supplements: {supplement}")
            if len(supplement) > 0:
                g.extend(supplement)
    
    ## 変数の追加
    #dfx.loc[:,target_var] = ngrams_inner
    dfx[target_var] = ngrams_inner
    
    ## check result
    print(dfx[target_var])

In [ ]:
## regular n-grams
for i in range(2, 6):
    print(f"adding {i}-gram column")
    add_ngram_to_df(df, n_for_ngram = i, skippy = False, inclusive = True, check = False)

In [ ]:
## skippy n-grams
for i in range(2, 6):
    print(f"adding skippy {i}-gram column")
    add_ngram_to_df (df, n_for_ngram = i, skippy = True, skippy_means_extended = True, max_distance = max_gap_val, inclusive = True, check = False)

In [ ]:
## store original index of df for later recovery
df_original_index = df.index
#df.sample(10)

# Analysis, Part 0: Building DTM

In [ ]:
## (re)define term_type
## term type: skippy-{4,5}-grams takes much time to generate
term_types = [ '1gram',
              '2gram', '3gram', '4gram', '5gram',
              'skippy2gram', 'skippy3gram', 'skippy4gram', 'skippy5gram']

redefine_term_type = True
if redefine_term_type:
    term_type  = term_types[1]
    print(f"term_type is redefined to {term_type}")
else:
    print(f"term_type {term_type} is not redefined")

## build explanatory variables
explanatory_var = term_type

## select bots for DTM
print(f"explanatory_var: {source_type} {explanatory_var}{hash_status}")
bots = df[explanatory_var]
bots

In [ ]:
## get term list
from collections import defaultdict
term_dict = defaultdict(int)
for bot in bots:
    for term in bot:
        term_dict[term] += 1
terms = sorted (list(term_dict.keys()), key = lambda x: len(x), reverse = True)
## inspection
import random
random.sample(terms, 10)

In [ ]:
## install multiprocess on necesssity
#!conda install multiprocess -y

In [ ]:
## build DTM: takes a few minutes to generate skippy{4,5}gram
import os
n_cores = max(1, int(os.cpu_count()/2)) # uses half cores to prevent memory shortage
print(f"building DTM using {term_type}{hash_status} for terms under {n_cores} cores...")
import multiprocess as mp
from itertools import product
with mp.Pool(n_cores) as pool:
    R = pool.starmap(lambda t, b: any(list(map(lambda x: t in x, b))), product(terms, bots))

## reshape R for DataFrame creation
import numpy as np
L = np.reshape(np.array(R), (len(terms), -1))

## create DataFrame
#dtm_df = pd.DataFrame(L, index = df_original_index).T # fails
dtm_df = pd.DataFrame(L, index = terms).T # transposition needed

## convert values
if use_numerical_encoding:
    dtm_df = dtm_df.apply(lambda x: x.map({True: 1, False: 0}), axis = 1)
##
dtm_df

In [ ]:
## recover original index
dtm_df = dtm_df.set_index(df_original_index)
dtm_df

In [ ]:
## reduce DTM by discarding low-frequency terms
if reduce_DTM:
    print(f"reducing DTM by filtering terms with frequency less than {min_term_freq}")
    dfx = dtm_df.copy()
    #
    size0 = dfx.shape[1]
    print(f"original column size: {size0}")

    dfx = dfx.loc[:, (dfx.sum(axis = 0) >= min_term_freq)]
    #
    size1 = dfx.shape[1]
    print(f"reduced column size: {size1}")
    print(f"{size0 - size1} columns are discarded")
    dtm_df = dfx

## make a coopy
dtm_df_original = dtm_df.copy()

# Analysis, Part 1: set up input and output

In [ ]:
## (re)define target attribute
target_attribs  = [ 'gender', 'plurality', 'case' ]
target_attrib   = target_attribs[0]
print(f"target_attrib: {target_attrib}")
##
if target_attrib == 'gender':
    ## Czech
    if target_lang in [ 'Czech' ]:
        if merge_cases:
            target_cols = [ 'Fem', 'Masc', 'Neut' ]
        else:
            target_cols = [ 'Fem', 'Masc0', 'Masc1', 'Neut' ]
    ## German
    elif target_lang in [ 'German' ]:
        target_cols    = [ 'Fem', 'Masc', 'Neut' ]
    ## French
    elif target_lang in [ 'French' ]:
        ignore_Comm = False
        if ignore_Comm:
            target_cols = [ 'Fem', 'Masc' ]
        else:
            target_cols = [ 'Fem', 'Masc', 'Comm' ]
    ## Others
    else:
        target_cols  = [ 'Fem', 'Masc' ]
##
elif target_attrib == 'plurality':
    ## French
    if target_lang in [ 'French' ]:
        target_cols  = [ 'Sg', 'Pl', 'Inv' ]
    ## Others
    else:
        target_cols  = [ 'Sg', 'Pl' ]
elif target_attrib == 'case':
    ignore_vocative = True
    if target_lang in [ 'Czech' ]:
        if ignore_vocative:
            target_cols = [ 'Nom', 'Acc', 'Dat', 'Gen', 'Instr', 'Loc' ]
        else:
            target_cols = [ 'Nom', 'Acc', 'Dat', 'Gen', 'Instr', 'Loc', 'Voc' ]
    elif target_lang in [ 'German' ]:
        if ignore_vocative:
            target_cols = [ 'Nom', 'Acc', 'Dat', 'Gen' ]
        else:
            target_cols = [ 'Nom', 'Acc', 'Dat', 'Gen', 'Voc' ]
    elif target_lang in [ 'Irish' ]:
        if ignore_vocative:
            target_cols = [ 'Nom', 'Dat', 'Gen' ]
        else:
            target_cols = [ 'Nom', 'Dat', 'Gen', 'Voc' ]
    else:
        print(f"Attribute {target_attrib} not defined for {target_lang}")
        raise UnboundLocalError
##
target             = df[target_cols]
target

In [ ]:
## remove offensive lines
original_len = len(target)
original_len == len(dtm_df)
offensive_indices = target[target.sum(axis = 1) == 0].index
print("indices of offensive rows")
offensive_indices
target = target.drop (offensive_indices)
dtm_df = dtm_df_original.drop (offensive_indices) # dtm_df cannot be used here
print (f"{len(offensive_indices)} rows are offensive and removed")

In [ ]:
## add Inv if plurality is target
if target_attrib in [ 'plurality' ]:
    try:
        target['Inv']
    except KeyError:
        target['Inv'] = [ 1 if x > 1 else 0 for x in (target['Sg'] + target['Pl']) ]
    ## change original values
    target.loc[:,'Sg'] = np.where(target['Inv'] == 1, 0, target.Sg)
    target.loc[:,'Pl'] = np.where(target['Inv'] == 1, 0, target.Pl)
    ##
    target_cols = target.columns
##
target

In [ ]:
## define label_to_int
label_to_int = { name: i for i, name in enumerate (sorted (target_cols)) }
label_to_int

In [ ]:
## define supplement
supplement_cols = [ x for x in N_attributes if not x in target_cols ]
supplement = df[supplement_cols]
supplement

In [ ]:
## use supplement or not
use_supplement   = False
if use_supplement:
    encoded = dtm_df.join (supplement)
else:
    encoded = dtm_df
##
encoded

# Set up for cross-validation

In [ ]:
## define training and test sets
from sklearn.model_selection import train_test_split
test_size_rate = 0.1
X_train, X_test, y_train, y_test = \
	train_test_split (encoded, target, test_size = test_size_rate, random_state = 0)

In [ ]:
X_train

In [ ]:
y_train

In [ ]:
## create singlified versions of y_ variables
import utils
reload_module = True
if reload_module:
    from importlib import reload
    reload (utils)
##
labels = list (y_train.columns)
print (f"labels to use: {labels}")
failure_mark = 'xxx'
y_train_single = utils.singlify_labels (y_train, labels, failure_mark = failure_mark, check = False)
y_test_single  = utils.singlify_labels (y_test, labels, failure_mark = failure_mark, check = False)

# Decision Tree Analysis

In [ ]:
## run DT Analysis
from sklearn import tree
max_depth = 20
dt_model = tree.DecisionTreeClassifier(max_depth = max_depth, random_state = 0, criterion = 'gini')
dt_fitting = dt_model.fit(X_train, y_train)
dt_fitting_single = dt_model.fit(X_train, y_train_single)

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
# use random search to find the best hyperparameters
dt = tree.DecisionTreeClassifier ()
dt_search_params = { 'max_features': randint(10, 120), 'max_depth': randint(2, 40)}
dt_rand_search = RandomizedSearchCV (dt, param_distributions = dt_search_params, n_iter = 5, cv = 5)

# fit the random search object to the data
dt_rand_search.fit(X_train, y_train_single)

# determine variables for the best model
best_dt = dt_rand_search.best_estimator_
print('best hyper-parameters:', dt_rand_search.best_params_)
# extract best parameter values
best_max_depth = dt_rand_search.best_params_['max_depth']
best_n_estimators = dt_rand_search.best_params_['max_features']

In [ ]:
## evaluate DT model
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import ConfusionMatrixDisplay
#dt_predict = dt_fitting.predict(X_test)
best_dt_predict = best_dt.predict(X_test)
## Score report is compatible with multi-labels
print(f"\nDT classification of {target_attrib} [supplement: {use_supplement}] of {target_lang} using {source_type} {term_type}{hash_status}")
print(classification_report(y_test_single, best_dt_predict, zero_division = 0.0))
## Confusion matrix: accepts only single-label output
dt_cm = confusion_matrix(y_test_single, best_dt_predict)
print(f"Confusion matrix of a DT classification of {target_attrib} [supplement: {use_supplement}] in {target_lang} using {source_type} {term_type}{hash_status}")
ConfusionMatrixDisplay(dt_cm).plot()
print(dt_cm)

In [ ]:
## plot Decision Tree result
import matplotlib.pyplot as plt
#draw_trees = False
if draw_trees:
    plt.figure(figsize = (130, 60))
    ax = plt.axes()
    #figure = plt.figure()
    tree.plot_tree(best_dt, feature_names = X_train.columns, class_names = y_train_single, filled = True, fontsize = 12) # gives unusual coloring
    #tree.plot_tree (dt_fitting_single, feature_names = X_train.columns, class_names = y_train_single, filled = True, fontsize = 12)
    title = f"DT classification of {target_attrib} in {target_lang} using {explanatory_var}{hash_status}"
    plt.title(title)

# Random Forest Analysis

In [ ]:
## run Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
# use random search to find the best hyperparameters
rf = RandomForestClassifier ()
rf_searchc_params = { 'n_estimators': randint(10, 120), 'max_depth': randint(2, 40)}
rf_rand_search = RandomizedSearchCV (rf, param_distributions = rf_searchc_params, n_iter = 5, cv = 5)
# fit the random search object to the data
rf_rand_search.fit(X_train, y_train_single)
# determine variables for the best model
best_rf = rf_rand_search.best_estimator_
print('best hyper-parameters:', rf_rand_search.best_params_)
# extract best parameter values
best_max_depth = rf_rand_search.best_params_['max_depth']
best_n_estimators = rf_rand_search.best_params_['n_estimators']

In [ ]:
## evaluate RF model
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import ConfusionMatrixDisplay
rf_predict = best_rf.predict(X_test)
##
print(f"\nRF (max_depth: {best_max_depth}, n_estimators: {best_n_estimators}) classification of {target_attrib} [supplement: {use_supplement}]  using: {source_type} {term_type}{hash_status}")
print(classification_report(y_test_single, rf_predict, zero_division = 0))
##
print(f"Confusion matrix of a RF (max_depth: {best_max_depth}, n_estimators: {best_n_estimators}) of {target_attrib} [supplement: {use_supplement}] in {target_lang} using: {source_type} {term_type}{hash_status}")
rf_cm = confusion_matrix(y_test_single, rf_predict)
ConfusionMatrixDisplay(rf_cm).plot()
print(rf_cm)

In [ ]:
## plot Random Forest results
import matplotlib.pyplot as plt
if draw_trees:
    plt.figure(figsize = (130, 60))
    tree.plot_tree (best_rf.estimators_[0], feature_names = X_train.columns, class_names = y_train_single, filled = True, fontsize = 12)
    title = f"Random Forest (max_depth: {best_max_depth}, n_estimators: {best_n_estimators}) classification of {target_attrib} [supplement: {use_supplement}] in {target_lang} using {explanatory_var}{hash_status}"
    plt.subtitle = title

# Analysis: NN Analysis

In [ ]:
## install keras on necessity
#!conda install keras TensorFlow -y

In [ ]:
## build NN model
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation
from keras.optimizers import SGD, Adam
##
NN_model = Sequential()
## settings
input_size  = X_train.shape[1]
print(f"input_size: {input_size}")
output_size = y_train.shape[1]
print(f"output_size: {output_size}")

k = 1.5
base_n        = round (k * math.sqrt(input_size))
dropout_rate  = 0.1
## activation for input and hidden layers
activation_funcs         = [ 'sigmoid', 'tanh', 'relu', 'softmax' ]
activation_func          = activation_funcs[1]
print(f"activation_func: {activation_func}")

## activation for output layer
output_activation_func   = activation_funcs[1]
print(f"output_activation_func: {output_activation_func}")

## input layer
NN_model.add (Dense(base_n, activation = activation_func, input_dim = input_size))
NN_model.add (Dropout(dropout_rate))

## hidden layer
n_hidden_layers = 3
divider         = 3

layer_ids = range (1, n_hidden_layers + 1)
down_sized_layers = [2,4,6,8]
for i in layer_ids:
    if i in down_sized_layers:
        n_units = int(round (base_n / divider, 0))
        print (f"adding {n_units} units at hidden layer {i}")
    else:
        n_units = base_n
        print (f"adding {n_units} units at hidden layer {i}")
    #
    NN_model.add (Dense(input_dim = input_size, units = n_units))
    NN_model.add (Activation(activation_func))
    NN_model.add (Dropout(dropout_rate))

## output layer
forced_choice = True
if target_attrib in [ 'gender', 'plurality', 'case' ]:
    if forced_choice:
        output_activation_func = 'softmax'
    else:
        output_activation_func = activation_func
print(f"output_activation_func is reset to: {output_activation_func}")
NN_model.add (Dense(output_size, activation = output_activation_func))

##
lr_val = 0.01
use_Adam = False
adam = Adam(learning_rate = lr_val)
sgd  = SGD (learning_rate = lr_val, decay = 1e-6, momentum = 0.9, nesterov = True)
if use_Adam:
    #NN_model.compile (loss = 'binary_crossentropy', optimizer = adam, metrics = ['accuracy'])
    NN_model.compile (loss = 'categorical_crossentropy', optimizer = adam, metrics = ['accuracy'])
else:
    #NN_model.compile (loss = 'binary_crossentropy', optimizer = sgd, metrics = ['accuracy'])
    NN_model.compile (loss = 'categorical_crossentropy', optimizer = sgd, metrics = ['accuracy'])

In [ ]:
## train NN model: requires Python 3.11 or later to complete
NN_model.fit (X_train, y_train, epochs = 150, verbose = 0)

In [ ]:
## generate NN_predict
print(f"output_activation_func: {output_activation_func}")
NN_predict = NN_model.predict (X_test)
#random.sample(list(NN_predict), 10)

## value conversion on NN predict
if output_activation_func == 'softmax':
    threshold = 1/len(target.columns)
    NN_predict [ NN_predict >= threshold ] = int(1)
    NN_predict [ NN_predict < threshold ]  = int(0)
else:
    if  output_activation_func == 'sigmoid':
        threshold = 0.5
        NN_predict [ NN_predict >= threshold ] = int(1)
        NN_predict [ NN_predict < threshold ]  = int(0)
    elif output_activation_func == 'tanh':
        threshold = 0
        NN_predict [ NN_predict >= threshold ] = int(1)
        NN_predict [ NN_predict < threshold ]  = int(0)
print(f"threshold set to: {threshold}")
#random.sample(list(NN_predict), 10)

In [ ]:
## evaluate NN
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import ConfusionMatrixDisplay
print(f"output_activation_func: {output_activation_func}")
if output_activation_func == 'softmax':
    ## convert distribution to determnisitc values using argmax
    NN_predict_converted = [ x.argmax() for x in NN_predict ]
    y_test_converted = [ x.argmax() for i, x in y_test.iterrows() ]
    ##
    print(f"\nNN classification of {target_attrib} [supplement: {use_supplement}] in {target_lang} using: {source_type} {term_type}{hash_status}")
    print(classification_report(y_test_converted, NN_predict_converted, zero_division = 0.0))
    ##
    print(f"Confusion matrix of a NN classification of {target_attrib} [supplement: {use_supplement}] in {target_lang} using {source_type} {term_type}{hash_status}")
    NN_cm = confusion_matrix (y_test_converted, NN_predict_converted)
    print(NN_cm)
    ConfusionMatrixDisplay(NN_cm).plot()
else:
    for i in range(len(target_cols)):
        test, predict = y_test.iloc[:,i], list(map(int, NN_predict[:,i]))
        ##
        print(f"\nNN classification of {target_attrib} [supplement: {use_supplement}] in {target_lang} for {target_cols[i]} using {source_type} {term_type}{hash_status}")
        print(classification_report(test, predict, zero_division = 0.0))
        ##
        print(f"Confusion matrix of a NN classification of {target_attrib} [supplement: {use_supplement}] in {target_lang} for {target_cols[i]} using {source_type} {term_type}{hash_status}")
        NN_cm = confusion_matrix (test, predict)
        print(NN_cm)
        ConfusionMatrixDisplay(NN_cm).plot()